# 03 · Clean a messy household survey

**Module:** 2 · **Time:** 45 minutes · **Prerequisites:** DataFrames, selecting, and filtering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/02_data_wrangling/03_cleaning.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F02_data_wrangling%2F03_cleaning.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** The repository is currently private and these badges refer to `main` after merge.

## Learning goals

- **S**tandardize inconsistent text labels without hiding raw values.
- **R**ecode sentinel missing values and convert numeric strings.
- **D**etect duplicate IDs and implausible wage observations.

In [ ]:
import pandas as pd

survey = pd.DataFrame({
    "person_id": [1001, 1002, 1003, 1003, 1005, 1006],
    "household_id": ["H001", "H001", "H002", "H002", "H003", "H003"],
    "region": [" North ", "SOUTH", "east", "north", "West", "SOUTH"],
    "employment": ["employed", "unemployed", "employed", "employed", "employed", "unemployed"],
    "wage_2023": ["40000", "-99", "51000", "52000", "72000", "not reported"],
    "wage_2024": ["42000", "28000", "53000", "54000", "9999999", "30000"],
    "survey_weight": [1.2, 0.8, 1.1, 1.4, 0.9, 1.6],
})
print(survey.head(2))

## Why this matters

Research data often arrive with `-99` standing for missing, mixed capitalization, duplicated person identifiers, and outliers. Cleaning should be explicit and auditable. The numbers here are invented for training; never silently remove observations from genuine datasets without documenting a rule.

## Keep raw columns and normalize strings

Create a copy, then add a new clean variable. This preserves the original labels for an audit. The `.str` accessor provides vectorized string cleaning.

In [ ]:
cleaned = survey.copy()
cleaned["region_clean"] = cleaned["region"].str.strip().str.lower()
cleaned["employment_clean"] = cleaned["employment"].str.strip().str.lower()
print(cleaned[["region", "region_clean"]].drop_duplicates())

## Convert numbers and missing codes

`errors='coerce'` turns invalid numeric text into missing data; known sentinel codes also need explicit replacement. Work on a new column rather than overwriting the raw survey.

In [ ]:
wage23 = pd.to_numeric(cleaned["wage_2023"], errors="coerce")
cleaned["wage_2023_numeric"] = wage23.mask(wage23.eq(-99))
print(cleaned[["wage_2023", "wage_2023_numeric"]])
print("Missing wages:", cleaned["wage_2023_numeric"].isna().sum())

## Flag impossible values and duplicate identifiers

A wage over 500,000 is a deliberately chosen **toy threshold**, not a universal rule. Flag unusual observations before choosing whether to exclude them. Preserve duplicated IDs for investigation.

In [ ]:
wage24 = pd.to_numeric(cleaned["wage_2024"], errors="coerce")
cleaned["wage_2024_flag"] = wage24.gt(500_000)
cleaned["duplicate_person_id"] = cleaned.duplicated("person_id", keep=False)
print(cleaned.loc[cleaned["wage_2024_flag"], ["person_id", "wage_2024"]])
print(cleaned.loc[cleaned["duplicate_person_id"], "person_id"])

## Categorical values and a cleaning audit

The `category` dtype records a finite set of labels. Summarize row counts and missing values and check any special exclusion decision before analysis.

In [ ]:
cleaned["employment_cat"] = cleaned["employment_clean"].astype("category")
print(cleaned["employment_cat"].value_counts())
assert len(cleaned) == len(survey)
assert cleaned["wage_2023_numeric"].isna().sum() >= 1
print("Cleaning audit: no rows silently dropped.")

## If you know Stata/R

| Task | Python / pandas | Stata | R |
| --- | --- | --- | --- |
| Trim text | `s.str.strip()` | `strtrim(s)` | `trimws(s)` |
| Recode missing | `s.mask(s.eq(-99))` | `replace x = . if x == -99` | `dplyr::na_if(x, -99)` |
| Convert numeric | `pd.to_numeric(s, errors="coerce")` | `destring x, replace` | `readr::parse_number(x)` (review warnings) |
| Find duplicate keys | `df.duplicated("id", keep=False)` | `duplicates tag id` | `duplicated(df$id)` (first duplicate only) |

Note that R's `duplicated` marks later duplicates; pandas `keep=False` marks every member of the duplicated group.

## Try it

Complete each blank answer cell, then run its friendly check. In the learner version, checks print hints instead of raising an exception.

### ★ Exercise 1 — Normalize region labels

Store a Series `region_cleaned` with stripped, lowercased values from `survey['region']`.

In [ ]:
region_cleaned = None

In [ ]:
expected = survey["region"].str.strip().str.lower()
if isinstance(region_cleaned, pd.Series) and region_cleaned.equals(expected):
    print("✓ Region labels standardized")
else:
    print("Hint: use the .str.strip() and .str.lower() accessors.")

### ★★ Exercise 2 — Recode missing 2023 wages

Create `wage23_clean` as numeric values, turning `-99` and invalid text into missing values.

In [ ]:
wage23_clean = None

In [ ]:
expected = pd.to_numeric(survey["wage_2023"], errors="coerce")
expected = expected.mask(expected.eq(-99))
if isinstance(wage23_clean, pd.Series) and wage23_clean.equals(expected):
    print("✓ Missing wages recoded")
else:
    print("Hint: use to_numeric(..., errors='coerce'), then mask -99.")

### ★★ Exercise 3 — Find every duplicated person ID

Create `duplicate_rows` containing all observations whose `person_id` appears more than once.

In [ ]:
duplicate_rows = None

In [ ]:
expected = survey.loc[survey.duplicated("person_id", keep=False)]
if isinstance(duplicate_rows, pd.DataFrame) and duplicate_rows.equals(expected):
    print("✓ Duplicated IDs identified")
else:
    print("Hint: duplicated(..., keep=False) flags all members of a duplicate group.")

### ★★★ Exercise 4 — Flag implausible 2024 wages

Create `large_wage_flag` as a Boolean Series for numeric 2024 wages strictly above 500,000.

In [ ]:
large_wage_flag = None

In [ ]:
expected = pd.to_numeric(survey["wage_2024"], errors="coerce").gt(500_000)
if isinstance(large_wage_flag, pd.Series) and large_wage_flag.equals(expected):
    print("✓ Outlier flag correct")
else:
    print("Hint: convert wage strings to numbers, then compare with the threshold.")

### ★★★ Exercise 5 — Create an employment category

Convert the original `employment` Series to the pandas categorical type and store it as `employment_cat`.

In [ ]:
employment_cat = None

In [ ]:
expected = survey["employment"].astype("category")
if isinstance(employment_cat, pd.Series) and employment_cat.equals(expected):
    print("✓ Employment categories ready")
else:
    print("Hint: Series.astype('category') returns a categorical Series.")

## Common errors

- `ValueError` converting `not reported` directly to a number — use `pd.to_numeric(..., errors='coerce')` after deciding how to treat invalid strings.
- Comparing `NaN` to itself with `==` gives unexpected results; use `isna()` to identify missing values.
- One duplicate ID is not automatically a duplicate observation; investigate your unit of analysis before dropping rows.
- Using a single outlier cutoff for all datasets can discard valid extreme values; document substantive reasoning before exclusions.

## Recap

- Preserve raw columns and document every cleaning decision.
- Use `to_numeric` and a missing-code rule for dirty numeric strings.
- Flag duplicate identifiers and extreme values before deciding how to handle them.
- Validate row counts and missing-value patterns after cleaning.

## Next steps

Next: [Creating variables](04_creating_variables.ipynb) (planned). See the [pandas missing-data guide](https://pandas.pydata.org/docs/user_guide/missing_data.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md) and include the lesson path.